In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('../'))
from src.utils.utils import load_file

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn

In [2]:
most_frequent_ingredients = ['Salt', 'Sugar', 'Onion', 'Milk', 'Garlic clove', 'Egg', 'All purpose flour', 
                             'Ground cumin', 'Clove', 'Garlic', 'Paprika']

In [3]:
# Restaurant data with the 25 most frequent ingredients
#restaurant_data_df = load_file('../data/kaggle/Restaurant_frequent_ings.csv') 
restaurant_data_df = load_file("../data/kaggle/Restaurant_food_quantities.csv")
restaurant_data_df['Ingredient'] = restaurant_data_df['Ingredient'].str.capitalize()

# Filter restaurant data by most frequent ingredients
restaurant_freq_ings_df = (restaurant_data_df[restaurant_data_df['Ingredient'].isin(most_frequent_ingredients)].copy()) # type: ignore
#display(restaurant_freq_ings_df)
#restaurant_freq_ings_df = restaurant_data_df.copy()

In [4]:
# Convert DateTime column to datetime type
restaurant_freq_ings_df['DateTime'] = pd.to_datetime(restaurant_freq_ings_df['DateTime'])

# Extract date from DateTime 
restaurant_freq_ings_df['Date'] = restaurant_freq_ings_df['DateTime'].dt.date 

# Group by Day, Ingredient, and DayType
consumption_df = restaurant_freq_ings_df.groupby(['Date', 'Ingredient'])['Total Quantity'].sum().reset_index() 
#display(consumption_df)

# Pivot the DataFrame to get quantities per ingredient
pivot_df = consumption_df.pivot_table(index='Date', columns=['Ingredient'], values='Total Quantity', fill_value=0)
pivot_df.columns.name = None

daily_consumption_df = pivot_df
display(daily_consumption_df)

,All purpose flour,Clove,Egg,Garlic,Garlic clove,Ground cumin,Milk,Onion,Paprika,Salt,Sugar
Date,,,,,,,,,,,
2019-01-01,929.931552,0.125728,162.553210,19.849773,7.655641,8.549301,4499.592825,1484.133756,2.441247,162.974727,1631.504078
2019-01-02,639.083809,0.070579,106.969282,13.459318,5.637389,5.438357,2504.630903,967.236602,1.808771,99.851043,998.712828
2019-01-03,686.197168,0.055736,136.313515,15.462596,5.770011,7.178583,3165.079547,1187.706418,2.481908,119.860452,1159.243446
2019-01-04,593.965971,0.052449,101.307948,13.284490,5.189715,5.502097,2828.859424,960.674780,1.406736,97.104337,964.581753
2019-01-05,1240.207705,0.149708,222.873950,27.495572,11.245257,11.037599,6149.556462,1979.324147,3.414856,198.989110,2073.857236
...,...,...,...,...,...,...,...,...,...,...,...
2019-06-26,717.132995,0.061994,135.202785,14.064159,6.121430,6.973147,3323.987307,1043.812859,2.004435,114.141431,1204.836139
2019-06-27,1196.724644,0.129388,210.186011,31.728243,11.150744,11.076868,5301.639319,2067.531114,3.366290,213.521591,2315.424930
2019-06-28,1035.395998,0.114119,184.916852,25.140486,9.480805,9.716583,5191.161591,1859.617286,2.857067,186.391889,1948.319450


In [5]:
# Print JSON list
import json

# {"ingredientId": "string", "day": "25", month: "3", "year", "quantity": 25 }

# Define the ingredient IDs and measure units
ingredient_ids = {
    'Salt': '5ba6c6c1-d2b2-4e11-b5ab-92ecd03ba6ea',
    'Sugar': '5ca0c5cd-923a-410a-85fb-97c1716c97d3',
    'Milk': 'c388bb8c-08c1-4aee-9322-ea9e9fd2df2d',
    'Onion': 'b405114e-8665-45ea-bd75-b6380167003c',
    'Garlic clove': '9b77593e-792d-4e99-9040-d0718499b77b',
    'Egg': '09af5da4-6d2f-4bda-9e84-79472eb43420',
    'All purpose flour': '65441fba-8235-4cbe-90b0-186bfff2ff3d',
    'Ground cumin': 'd08192cd-571e-4b14-b2ee-cd1fad261834',
    'Clove': 'f8989c10-b9cf-4c29-94cd-f507560f654c',
    'Paprika': '8519c9c0-19e3-4242-b161-81c7213acbae',
    'Garlic': 'eac248d0-fb9d-4263-9399-6749ebfafde3',
}

# convert all quantities to kg 
daily_consumption_df = daily_consumption_df.astype(float) 
#display(daily_consumption_df)

# Update the year in the index
#forecast_quantities.index = forecast_quantities.index.map(lambda x: x.replace(year=2024))

# Convert DataFrame to list of JSON objects
json_list = []
for date, row in daily_consumption_df.iterrows():
    for ingredient, quantity in row.items():
        print("ingredient: ", ingredient)
        print("quantity: ", quantity)
        if(quantity < 1000):
            quantityInKg = round(quantity / 1000, 3)   # kg
            print("quantityInKg: ", quantityInKg, "\n")
            if quantityInKg == 0:
                quantityInKg = round(quantity / 1000, 5)   # kg
            print("quantityInKg_2: ", quantityInKg, "\n")
        else: 
            quantityInKg = round(quantity / 1000, 1)   # kg
            print("quantityInKg: ", quantityInKg, "\n")
        
        json_obj = {
            "ingredientId": ingredient_ids.get(ingredient, "unknown_id"),
            "day": int(date.strftime("%d")),
            "month": int(date.strftime("%m")),
            "year": int(date.strftime("%Y")),
            "quantity": quantityInKg,
        }
        json_list.append(json_obj)

# Print JSON list
#print(json.dumps(json_list, indent=2))

# Write JSON list to a file
with open('consumption_data.json', 'w') as json_file:
    json.dump(json_list, json_file, indent=2)

ingredient:  All purpose flour
quantity:  929.9315519974156
quantityInKg:  0.93 

quantityInKg_2:  0.93 

ingredient:  Clove
quantity:  0.1257275339894087
quantityInKg:  0.0 

quantityInKg_2:  0.00013 

ingredient:  Egg
quantity:  162.5532095093775
quantityInKg:  0.163 

quantityInKg_2:  0.163 

ingredient:  Garlic
quantity:  19.849772973155417
quantityInKg:  0.02 

quantityInKg_2:  0.02 

ingredient:  Garlic clove
quantity:  7.655640715615224
quantityInKg:  0.008 

quantityInKg_2:  0.008 

ingredient:  Ground cumin
quantity:  8.549300596910793
quantityInKg:  0.009 

quantityInKg_2:  0.009 

ingredient:  Milk
quantity:  4499.592824682376
quantityInKg:  4.5 

ingredient:  Onion
quantity:  1484.1337561083567
quantityInKg:  1.5 

ingredient:  Paprika
quantity:  2.4412470969979703
quantityInKg:  0.002 

quantityInKg_2:  0.002 

ingredient:  Salt
quantity:  162.97472726636627
quantityInKg:  0.163 

quantityInKg_2:  0.163 

ingredient:  Sugar
quantity:  1631.504077922099
quantityInKg:  1.6 
